In [1]:
import pandas as pd
import warnings
import platform
from sqlalchemy import create_engine
import pandas as pd
import numpy as np
import pymysql
import datetime as dt
import os
import json
import platform
from googleapiclient.discovery  import build
from google.oauth2 import service_account
import psycopg2
import pandas as pd
import numpy as np
import datetime 
import pymysql
import yagmail
import time
import platform
from sqlalchemy import create_engine
from googleapiclient.discovery import build
from google.oauth2 import service_account
from datetime import timedelta,datetime
from datetime import datetime
from dateutil.relativedelta import relativedelta
### SQL trying

import requests
import json
import pandas as pd
import time
import platform
import datetime
import psycopg2
import yagmail
from datetime import datetime
from dateutil.relativedelta import relativedelta
from io import StringIO
from sqlalchemy import create_engine

import traceback
from langchain_openai import ChatOpenAI
from sqlalchemy import create_engine,text
from dotenv import load_dotenv
import decimal
import math
import pytz
from datetime import datetime

# import gspread
# from oauth2client.service_account import ServiceAccountCredentials
from google.oauth2 import service_account
from googleapiclient.discovery import build


In [2]:
# FL = {'location': r"C:\Documents\Python_Script"}

FL = {'location': r"C:\Users\Amit Singh\Documents\Python_Scripts"}

In [3]:
#For Voylla DB Cred
if platform.system()=='Windows':
    with open(r"%s\Voylla_Cred.txt" % FL['location'],'r') as f:
        lines = f.readlines()
        lines = [item.strip() for item in lines]
        Voylla_config = {
            'user':  lines[1],
            'password': lines[2],
            'host': lines[0],
            'database': lines[3],
            'port': int(lines[4])
        }
    
elif platform.system()=='Linux':
    with open(r'/home/misauto/Python_Scripts/Voylla_Cred.txt') as f:
        lines = f.readlines()
        lines = [item.strip() for item in lines]
        Voylla_config = {
            'user':  lines[1],
            'password': lines[2],
            'host': lines[0],
            'database': lines[3],
            'port': int(lines[4])
        }

# for gpt api key
if platform.system()=='Windows':
    with open(r"%s\Gpt_api_key.txt" % FL['location'],'r') as f:
        lines = f.readlines()
        lines = [item.strip() for item in lines]
        Gpt_api = {
            'model_name':  lines[0],
            'api': lines[1]
        }
    
elif platform.system()=='Linux':
    with open(r'/home/misauto/Python_Scripts/Gpt_api_key.txt') as f:
        lines = f.readlines()
        lines = [item.strip() for item in lines]
        Gpt_api = {
            'model_name':  lines[0],
            'api': lines[1]
        }



In [4]:
from sqlalchemy import text
from datetime import datetime


LLM_TEMPERATURE = 0

# MODEL_NAME = "gpt-4.1-mini"

# Extract values

db_host = Voylla_config["host"]
db_port = Voylla_config["port"]
db_name = Voylla_config["database"]
db_user = Voylla_config["user"]
db_password = Voylla_config["password"]

api_key=Gpt_api["api"]
MODEL_NAME= Gpt_api["model_name"]


def get_llm():
    return ChatOpenAI(model=MODEL_NAME, temperature=LLM_TEMPERATURE, request_timeout=120, max_retries=3,api_key=api_key)

llm = get_llm()


# DB connection
def get_engine_and_schema():
    
    try:
        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}",
            pool_pre_ping=True,
            pool_recycle=3600,
            pool_size=5,
            max_overflow=10
        )

        with engine.connect() as conn:
            print("✅ Database connection successful")

        return engine


    except Exception as e:
        print(f"❌ Unexpected error while creating engine: {e}")
        return None



engine = get_engine_and_schema()
if engine is None:
    print("Stopping script due to DB failure")
    exit()


✅ Database connection successful


In [5]:
query = """
SELECT DISTINCT "Brand"
FROM voylla."Blinkit_Ads_Report"
"""

df = pd.read_sql(query, engine)

Brands = df["Brand"].dropna().unique().tolist()

print(Brands)

['Voylla', 'Petcrux']


In [6]:
import re
import json
def extract_json(response: str) -> list:
    """
    Extracts JSON array from LLM response.
    Handles scratchpad lines, markdown fences, and partial responses.
    """
    response = re.sub(r'```(?:json)?', '', response).strip()
    
    json_start = response.find('[')
    if json_start < 0:
        raise ValueError("No JSON array found in response")
    response = response[json_start:]
    
    json_end = response.rfind(']') + 1
    if json_end == 0:
        raise ValueError("No closing bracket found in response")
    response = response[:json_end]
    
    try:
        return json.loads(response)
    except json.JSONDecodeError as e:
        response_fixed = re.sub(r',\s*([}\]])', r'\1', response)
        try:
            return json.loads(response_fixed)
        except json.JSONDecodeError:
            raise ValueError(f"Could not parse JSON from response: {e}")

In [7]:
def aggregate_window(df, days):
    """
    Aggregate performance for given window
    """

    agg = df.groupby(['Campaign ID', 'Targeting Value']).agg({
        'spend': 'sum',
        'total_sales': 'sum',
        'impressions': 'sum',
        'total_atc': 'sum',
        'total_units': 'sum',
        'most_viewed_position': 'median',
        'Campaign Name': 'last'
    }).reset_index()

    agg[f'roas_{days}d'] = (agg['total_sales'] / agg['spend']).replace(
        [float('inf'), -float('inf')], 0
    ).fillna(0).round(2)

    agg[f'ctr_{days}d'] = (agg['total_atc'] / agg['impressions']).fillna(0).round(4)

    agg = agg.rename(columns={
        'Campaign ID': 'campaign_id',
        'Targeting Value': 'targeting',
        'most_viewed_position': 'position',
        'spend': f'spend_{days}d',
        'total_sales': f'sales_{days}d'
    })

    return agg


In [8]:
import pandas as pd

def resolve_impl(user_impl):
    if user_impl is None:
        return "Unknown"
    
    # handle string values from DB
    if isinstance(user_impl, str):
        val = user_impl.strip().lower()
        if val == 'true':
            return "Implemented"
        elif val == 'false':
            return "Not implemented"
        else:
            return "Unknown"
    
    try:
        if pd.isna(user_impl):
            return "Unknown"
    except (TypeError, ValueError):
        pass
    
    return "Implemented" if bool(user_impl) else "Not implemented"

def build_previous_context(history_df, campaign_id_str, targeting_key):
    filtered = history_df[
        (history_df["campaign_id"].astype(str) == campaign_id_str) &
        (history_df["targeting"].astype(str)
             .str.strip().str.lower().str.replace(" ", "_")
         == targeting_key.strip().lower().replace(" ", "_"))
    ].sort_values("action_date", ascending=False)

    if filtered.empty:
        return {
            "previous_summary": "No previous recommendation — fresh cycle.",
            "previous_history": []
        }

    rec       = filtered.iloc[0].to_dict()
    action    = rec.get("action", "UNKNOWN")
    date      = str(rec.get("action_date", ""))
#     raw_val=rec.get("user_implemented")
    impl_str  = resolve_impl(rec.get("user_implemented"))

    override  = rec.get("override_note") or ""
    note_str  = f" | Note: {override}" if override else ""

    summary = f"[{date}] {action} | {impl_str}{note_str}"

    history_rows = [
        {
            "date":          str(r.get("action_date", "")),
            "action":        r.get("action", "UNKNOWN"),
            "implemented":   resolve_impl(r.get("user_implemented")),
            "override_note": r.get("override_note") or ""
        }
        for r in filtered.head(5).to_dict(orient="records")
    ]

    return {
        "previous_summary": summary,
        "previous_history": history_rows
    }

In [ ]:
for brand in Brands:
    
    
    from sqlalchemy import text
    from datetime import datetime

    def save_llm_action(engine, action_obj, brand):

        if not action_obj:
            print("⚠️ No rows to insert")
            return

        if isinstance(action_obj, dict):
            action_obj = [action_obj]

        clean_rows = []

        for a in action_obj:

            if not isinstance(a, dict):
                continue

            campaign_id   = a.get("campaign_id")
            campaign_name = a.get("campaign_name", "")
            targeting     = str(a.get("targeting", "")).strip().lower().replace(" ", "_")
            action        = str(a.get("action", "")).upper()
            explanation   = a.get("explanation", "")

            confidence = a.get("confidence")

            try:
                confidence = float(str(confidence).strip())
            except:
                print(f"⚠️ Missing/invalid confidence → skipping row {targeting}")
                continue

            cpm_change = a.get("cpm_change")
            try:
                cpm_change = int(cpm_change)
            except:
                cpm_change = None
                
            current_cpm = a.get("current_cpm")
            if current_cpm is None:
                current_cpm = None
            else:
                try:
                    current_cpm = float(current_cpm)
                    if math.isnan(current_cpm):
                        current_cpm = None
                except:
                    current_cpm = None
                    

            campaign_budget = a.get("campaign_budget")

            try:
                campaign_budget = float(campaign_budget) if campaign_budget is not None else None
            except:
                campaign_budget = None

            if current_cpm is None:
                explanation = explanation.replace("CPM ₹nan", "CPM unavailable")
                explanation = explanation.replace("CPM ₹NaN", "CPM unavailable")

            alt = a.get("alternative_keywords")
            if not isinstance(alt, list):
                alt = []

            action_date = a.get("action_date")
            if not action_date:
                action_date = datetime.now().date()

            date_str = action_date.strftime("%Y-%m-%d")


            unique_key = f"{campaign_id}_{date_str}_{targeting}_{action}"

            clean_rows.append({
                "unique_key": unique_key,
                "action_date": action_date,
                "campaign_id": campaign_id,
                "campaign_name": campaign_name,
                "targeting": targeting,
                "action": action,
                "cpm_change": cpm_change,
                "confidence": confidence,
                "explanation": explanation,
                "alternative_keywords": alt,
                "Brand": brand,
                "current_cpm": current_cpm,
                "campaign_budget":campaign_budget
            })

        if not clean_rows:
            print("⚠️ No valid rows after cleaning")
            return

        insert_sql = text("""
            INSERT INTO voylla."Blinkit_actions_llm"
            (unique_key, action_date, campaign_id, campaign_name,
             targeting, action, bid_change, confidence,
             explanation, alternative_keywords, "Brand", current_cpm, campaign_budget)
            VALUES
            (:unique_key, :action_date, :campaign_id, :campaign_name,
             :targeting, :action, :cpm_change, :confidence,
             :explanation, :alternative_keywords, :Brand, :current_cpm, :campaign_budget)

            ON CONFLICT (unique_key) DO UPDATE SET
                bid_change = EXCLUDED.bid_change,
                confidence = EXCLUDED.confidence,
                explanation = EXCLUDED.explanation,
                alternative_keywords = EXCLUDED.alternative_keywords,
                current_cpm = EXCLUDED.current_cpm,
                campaign_budget = EXCLUDED.campaign_budget;
        """)

        with engine.begin() as conn:
            conn.execute(insert_sql, clean_rows)

        print(f"✅ Inserted/updated {len(clean_rows)} rows")

    
    query=f"""
    
    WITH base AS (
        SELECT
            TO_TIMESTAMP(a."Date", 'YYYY-MM-DD HH24:MI:SS')::date AS report_date,
            a."Campaign ID",
            a."Campaign Name",
            a."Targeting Type",
            a."Targeting Value",
            a."Match Type",
    
            SUM(a."Impressions") AS impressions,
            SUM(a."Direct ATC" + a."Indirect ATC") AS total_atc,
            SUM(a."Direct Quantities Sold" + a."Indirect Quantities Sold") AS total_units,
            SUM(a."Direct Sales" + a."Indirect Sales") AS total_sales,
            SUM(a."Estimated Budget Consumed") AS spend,
    
            AVG(a."CPM") AS avg_cpm,
            MAX(a."Pacing Type") AS pacing_type,
            MAX(a."Most Viewed Position") AS most_viewed_position
    
        FROM voylla."Blinkit_Ads_Report" a
        WHERE TO_TIMESTAMP(a."Date", 'YYYY-MM-DD HH24:MI:SS')
              >= CURRENT_DATE - INTERVAL '31 days' 
         AND a."Brand" = '{brand}'
        GROUP BY
            report_date,
            a."Campaign ID",
            a."Campaign Name",
            a."Targeting Type",
            a."Targeting Value",
            a."Match Type"
    ),
    
    metrics AS (
        SELECT *,
            CASE WHEN impressions > 0
                 THEN total_atc::FLOAT / impressions
            END AS ctr,
    
            CASE WHEN total_atc > 0
                 THEN total_units::FLOAT / total_atc
            END AS cvr,
    
            CASE WHEN spend > 0
                 THEN total_sales / spend
            END AS roas
        FROM base
    )
    
    SELECT
        *,
    
        AVG(roas) OVER (
            PARTITION BY "Campaign ID", "Targeting Value"
            ORDER BY report_date
            RANGE BETWEEN INTERVAL '7 day' PRECEDING
                  AND INTERVAL '1 day' PRECEDING
        ) AS roas_7d_avg,
    
        AVG(roas) OVER (
            PARTITION BY "Campaign ID", "Targeting Value"
            ORDER BY report_date
            RANGE BETWEEN INTERVAL '15 day' PRECEDING
                  AND INTERVAL '1 day' PRECEDING
        ) AS roas_15d_avg,
    
        AVG(roas) OVER (
            PARTITION BY "Campaign ID", "Targeting Value"
            ORDER BY report_date
            RANGE BETWEEN INTERVAL '30 day' PRECEDING
                  AND INTERVAL '1 day' PRECEDING
        ) AS roas_30d_avg,
    
    
        AVG(ctr) OVER (
            PARTITION BY "Campaign ID", "Targeting Value"
            ORDER BY report_date
            RANGE BETWEEN INTERVAL '7 day' PRECEDING
                  AND INTERVAL '1 day' PRECEDING
        ) AS ctr_7d_avg,
    
    
        roas -
        AVG(roas) OVER (
            PARTITION BY "Campaign ID", "Targeting Value"
            ORDER BY report_date
            RANGE BETWEEN INTERVAL '30 day' PRECEDING
                  AND INTERVAL '1 day' PRECEDING
        ) AS roas_vs_30d
    
    FROM metrics;
    """
    
    df=pd.read_sql(query,engine)
    df['report_date'] = pd.to_datetime(df['report_date'])
    today = df['report_date'].max()
#     today = pd.to_datetime("2026-03-25")
    
    df7  = df[df['report_date'] >= today - pd.Timedelta(days=7)]
    df15 = df[df['report_date'] >= today - pd.Timedelta(days=15)]
    df30 = df[df['report_date'] >= today - pd.Timedelta(days=30)]

    agg7  = aggregate_window(df7, 7)
    agg15 = aggregate_window(df15, 15)
    agg30 = aggregate_window(df30, 30)    

    aggregated_df = agg7.merge(
        agg15[['campaign_id','targeting','spend_15d','roas_15d','ctr_15d']],
        on=['campaign_id','targeting'],
        how='left'
    ).merge(
        agg30[['campaign_id','targeting','spend_30d','roas_30d','ctr_30d']],
        on=['campaign_id','targeting'],
        how='left'
    )    

    aggregated_df = aggregated_df[
    aggregated_df["spend_7d"] > 0]
    
    
    cpm_query=f"""
    select * from voylla."Blinkit_CPM" a
    where a."Brand"='{brand}' 
    """;
    
    cpm_df = pd.read_sql(cpm_query, engine)

    cpm_df = cpm_df.rename(columns={
        "Campaign ID": "campaign_id",
        "keyword": "targeting"
    })
    
    aggregated_df["campaign_id"] = aggregated_df["campaign_id"].astype(str)
    cpm_df["campaign_id"] = cpm_df["campaign_id"].astype(str)
    aggregated_df = aggregated_df.merge(
    cpm_df,
    on=["campaign_id", "targeting"],
    how="left"
    )
    
    
    budget_query = f"""
    SELECT 
        campaign_id,
        campaign_budget,
        status,
        created_at
    FROM voylla."Blinkit_CampaignWise_ProductID" where brand_name='{brand}'
    """

    budget_df = pd.read_sql(budget_query, engine)
    
    budget_df = budget_df.sort_values(by="created_at", ascending=False)
    budget_df = budget_df.drop_duplicates(subset=["campaign_id"], keep="first")
    budget_df = budget_df.drop(columns=["created_at"])
    
    budget_df["campaign_id"] = budget_df["campaign_id"].astype(str).str.strip()
    aggregated_df["campaign_id"] = aggregated_df["campaign_id"].astype(str).str.strip()
    
    aggregated_df = aggregated_df.merge(
        budget_df,
        on="campaign_id",
        how="left"
    )

    campaign_query = f"""
    SELECT DISTINCT "Campaign ID"
    FROM voylla."Blinkit_Ads_Report"
    WHERE TO_TIMESTAMP("Date",'YYYY-MM-DD HH24:MI:SS')
          >= CURRENT_DATE - INTERVAL '7 days'
          AND "Brand" = '{brand}' ;
    """
    
    campaign_df = pd.read_sql(campaign_query, engine)
    
    campaign_ids = campaign_df["Campaign ID"].tolist()
    
    print(campaign_ids)
    
    campaign_name_map = (
        df[["Campaign ID", "Campaign Name"]]
        .drop_duplicates()
        .set_index("Campaign ID")["Campaign Name"]
        .to_dict()
    )

    
    history_query=f"""
    SELECT
        unique_key,
        campaign_id,
        campaign_name,
        targeting,
        action,
        bid_change,
        confidence,
        explanation,
        action_date,
        user_implemented, 
        override_note
    
    FROM voylla."Blinkit_actions_llm"
    Where "Brand" = '{brand}'  
    ORDER BY action_date DESC;
    """
    
    history_df=pd.read_sql(history_query,engine)

    history_df["campaign_id"] = history_df["campaign_id"].astype(str)
    aggregated_df["campaign_id"] = aggregated_df["campaign_id"].astype(str)

    
    SPEND_THRESHOLD = 500
    
    campaign_spend_query = text(f"""
        SELECT
        "Campaign ID",
        "Campaign Name",
        SUM("Estimated Budget Consumed") AS campaign_spend
        FROM voylla."Blinkit_Ads_Report" a
        WHERE TO_TIMESTAMP("Date",'YYYY-MM-DD HH24:MI:SS')
              >= CURRENT_DATE - INTERVAL '7 days'
              AND a."Brand" = '{brand}'
        GROUP BY
            "Campaign ID",
            "Campaign Name"
        ORDER BY campaign_spend DESC;
    """)
    
    
    with engine.connect() as conn:
        campaign_spend_df = pd.read_sql(campaign_spend_query, conn)
        
    
    campaign_spend_df["Campaign ID"] = (
        campaign_spend_df["Campaign ID"]
        .astype(str)
        .str.strip()
    )
    
    sufficient_campaigns = set(
        campaign_spend_df[
            campaign_spend_df["campaign_spend"] >= SPEND_THRESHOLD
        ]["Campaign ID"]
    )
    
    insufficient_campaigns = set(
        campaign_spend_df[
            campaign_spend_df["campaign_spend"] < SPEND_THRESHOLD
        ]["Campaign ID"]
    )
    
      
    
    print(f"✅ Sufficient campaigns (≥₹{SPEND_THRESHOLD}): {len(sufficient_campaigns)}")
    print(f"⚠️  Insufficient campaigns (<₹{SPEND_THRESHOLD}): {len(insufficient_campaigns)}")

    campaign_name_map_str = {str(k): v for k, v in campaign_name_map.items()}

    import json
    import re
    from langchain_core.messages import SystemMessage, HumanMessage
    
    
    # ============================================================
    # EXTRACT JSON UTILITY
    # ============================================================
    
    def extract_json(response: str) -> list:
        """
        Extracts JSON array from LLM response.
        Handles scratchpad lines, markdown fences, and partial responses.
        """
        response = re.sub(r'```(?:json)?', '', response).strip()
        
        json_start = response.find('[')
        if json_start < 0:
            raise ValueError("No JSON array found in response")
        response = response[json_start:]
        
        json_end = response.rfind(']') + 1
        if json_end == 0:
            raise ValueError("No closing bracket found in response")
        response = response[:json_end]
        
        try:
            return json.loads(response)
        except json.JSONDecodeError as e:
            response_fixed = re.sub(r',\s*([}\]])', r'\1', response)
            try:
                return json.loads(response_fixed)
            except json.JSONDecodeError:
                raise ValueError(f"Could not parse JSON from response: {e}")
    
    
    # ============================================================
    # SYSTEM PROMPT — stored once, reused every campaign run
    # ============================================================
    
    SYSTEM_PROMPT = """
    
    You are a senior performance marketing analyst specializing in quick commerce
    advertising on Blinkit. You have managed ₹1Cr+ in CPM keyword campaigns and
    understand the nuances of bid optimization, position dynamics, and ROAS
    protection in high-velocity commerce environments.
    
    Your decisions are data-driven, conservative, and fully traceable.
    Every action must be justified by exact numbers from the data provided.
    
    ═══════════════════════════════════════════════════════════════
    CAMPAIGN CONTEXT
    ═══════════════════════════════════════════════════════════════
    Platform         : Blinkit (quick commerce)
    Ad type          : CPM keyword targeting
    Weekly budget    : ₹500 per keyword
    Data windows     : 7-day (current), 15-day (trend), 30-day (historical baseline)
    Primary goal     : Maximize ROAS — conservative scaling, aggressive protection
    Run type         : RECURRING — prior recommendations evaluated every cycle
    
    ═══════════════════════════════════════════════════════════════
    CORE PHILOSOPHY
    ═══════════════════════════════════════════════════════════════
    1. Historical ROAS is your anchor. 30-day ROAS stripped of weekly noise is the
       single most reliable signal. Never let a 7-day dip override it.
    
    2. Data before decisions. An insufficient window cannot prove failure.
       Insufficient spend = no judgment. Period.
    
    3. PAUSE is a last resort, not a default response to underperformance.
       Pausing destroys position rank, resets spend learning, and is nearly
       impossible to recover from competitively. Use it only when all three windows
       confirm sustained failure with sufficient spend backing each signal.
    
    4. Position is an asset. A top-10 position took weeks of spend to earn.
       Never sacrifice it based on a single weak window.
    
    5. Strong history = floor, not ceiling. Historical strength protects against
       panic-cutting. It must never prevent scaling when all windows confirm it.
    
    Decision priority (highest to lowest):
      1. Data sufficiency tier — classified before any ROAS analysis
      2. 30-day ROAS (most reliable)
      3. 15-day ROAS (trend confirmation)
      4. 7-day ROAS (current snapshot — weakest signal alone)
      5. Previous recommendation outcome
    
    ═══════════════════════════════════════════════════════════════
    STEP 1 — TIER CLASSIFICATION (runs first, no exceptions)
    ═══════════════════════════════════════════════════════════════
    Classify every keyword using keyword-level spend (not campaign spend):
    
      Thresholds:
        7-day  : keyword_spend_7d  ≥ ₹500
        15-day : keyword_spend_15d ≥ ₹1,000
        30-day : keyword_spend_30d ≥ ₹2,000
    
      TIER 3 — ALL THREE WINDOWS INSUFFICIENT:
        ▸ THE ONLY VALID ACTION IS INCREASE_CPM. NO EXCEPTIONS WHATSOEVER.
        ▸ Do not evaluate ROAS. Do not check position.
        ▸ Do not check previous recommendations.
        ▸ PAUSE, NO_CHANGE, and DECREASE_CPM are all forbidden on TIER 3. DEREASE_CPM only if cpm>200.
        ▸ PAUSE on TIER 3 is never valid under any condition.
        ▸ Note ROAS signal in explanation if present (context only).
        ▸ EXIT immediately. Do not proceed to Step 2.
    
      TIER 2 — AT LEAST ONE WINDOW INSUFFICIENT:
        ▸ Use only spend-sufficient windows for ROAS analysis.
        ▸ PAUSE is forbidden. PAUSE requires TIER 1.
        ▸ Proceed to Step 2.
    
      TIER 1 — ALL THREE WINDOWS SUFFICIENT:
        ▸ Full data confidence. Proceed to Step 2.
    
      CRITICAL: Insufficiency = not enough data to judge this window.
      It is NOT evidence of underperformance. Never treat low ROAS in an
      insufficient window as a signal of failure.
    
    ═══════════════════════════════════════════════════════════════
    STEP 2 — HISTORY GATE (hard exit — fires before Step 3 rules)
    ═══════════════════════════════════════════════════════════════
    Check spend-sufficient windows only. Hard threshold: 3.0. Do not round down.
      3.29 = strong. 3.36 = strong. 2.99 = NOT strong.
    
      ── STRONG HISTORY ──────────────────────────────────────────
      Trigger: roas_30d ≥ 3.0 (30d sufficient) OR roas_15d ≥ 3.0 (15d sufficient)
    
      → HISTORY GATE FIRES. DECREASE_CPM and PAUSE permanently blocked.
      → Do NOT evaluate Step 3 rules. Gate has fired. Action is locked.
    
      Determine locked action:
        SCALE (all must be true):
          ✓ roas_7d > 4.0
          ✓ roas_15d > 3.5  (if 15-day sufficient)
          ✓ roas_30d > 3.0  (if 30-day sufficient)
          ✓ position > 10
          ✓ position > 5    (if ≤ 5 → HOLD instead)
          → Action: INCREASE_CPM by 10%
    
        HOLD (default when SCALE conditions not fully met):
          → Action: NO_CHANGE
          → Explanation must state exact reason scaling not triggered
    
      ── MODERATE HISTORY ────────────────────────────────────────
      Trigger: roas_30d 2.0–2.99 (sufficient) AND roas_15d < 3.0 (sufficient)
    
      → PAUSE forbidden. DECREASE_CPM allowed (max 10%) DEREASE_CPM only if cpm>200.
      → If previous recommendation for this keyword was FAILURE → NO_CHANGE instead.
      → Proceed to Step 3 with PAUSE permanently blocked.
    
      ── WEAK / NO HISTORY ───────────────────────────────────────
      Trigger: roas_30d < 2.0 AND roas_15d < 2.0 (sufficient windows)
    
      → No historical protection. Proceed to Step 3.
    
    ⚠️ HARD FLOOR RULE (applies globally, no exceptions):
   DECREASE_CPM is only valid when current_cpm > ₹200 (strictly greater than).
    ═══════════════════════════════════════════════════════════════
    STEP 3 — ROAS THRESHOLD RULES (TIER 1/2, gate did not fire)
    ═══════════════════════════════════════════════════════════════
    Apply first matching rule. Stop immediately at first match.
    
    
      RULE A NUMERIC CHECK (verify before proceeding to Rule B):
          If roas_7d < 1.0 AND roas_15d < 1.0 AND roas_30d < 1.0:
            → Rule A qualifies. Do NOT evaluate Rule B.
            → Rule B is only evaluated when roas_7d ≥ 1.0.
            → If roas_7d = 0.75, that is < 1.0. Rule B does not apply.
            → If roas_7d = 0.99, that is < 1.0. Rule B does not apply.
            → Only roas_7d = 1.0 or above reaches Rule B.
        
        WEAK HISTORY + ALL ROAS < 1.0 + TIER 1 + POSITION > 10:
          This is the one scenario where PAUSE fires. Do not route to DECREASE_CPM.
          Example: roas_7d=0.75, roas_15d=0.52, roas_30d=0.60, spend sufficient,
          position 168 → PAUSE. Not DECREASE_CPM.
            
    
      RULE B — DECREASE_CPM (efficiency correction):
          DEREASE_CPM only if cpm>200
          ✓ roas_7d ≥ 1.0 AND roas_7d < 3.0
          ✓ current_cpm > ₹200 (if ≤ ₹200 → NO_CHANGE instead)
          ✓ current_cpm MUST BE STRICTLY GREATER THAN ₹200
          (if current_cpm = ₹200 → NO_CHANGE. If current_cpm = ₹199 → NO_CHANGE.)
        → DECREASE_CPM by exactly 10%.
        MINIMUM CPM FLOOR = ₹200. DECREASE_CPM is INVALID at CPM ≤ ₹200.
        
    
      RULE C — NO_CHANGE (target performance):
          ✓ roas_7d ≥ 3.0 AND roas_7d ≤ 4.0
        → NO_CHANGE. Confidence 0.80.
    
      RULE D — INCREASE_CPM (conservative scale):
          ✓ roas_7d > 4.0
          ✓ roas_15d > 3.5  (if 15-day sufficient)
          ✓ roas_30d > 3.0  (if 30-day sufficient)
          ✓ position > 10
          ✓ position > 5    (if ≤ 5 → NO_CHANGE)
        → INCREASE_CPM by exactly 10%.
        
        Note: ₹200 floor blocks DECREASE_CPM only. INCREASE_CPM valid at any CPM.
    
    ═══════════════════════════════════════════════════════════════
    STEP 4 — POSITION PROTECTION GATE (final check before output)
    ═══════════════════════════════════════════════════════════════
    Run before writing any output. No exceptions.
    
      Position ≤ 10 AND chosen action = PAUSE:
        → WRONG. Override immediately:
            
        → DECREASE_CPM only if current_cpm > ₹200 (strictly). If current_cpm = ₹200 → NO_CHANGE.
    
          → DECREASE_CPM if roas_7d < 3.0 AND CPM > ₹200 AND no strong history
          → NO_CHANGE otherwise
          -> If position = 1, then don't increase CPM.
    
      Position ≤ 5:
        → PAUSE forbidden.
        
        → DECREASE_CPM only if current_cpm > ₹200 (strictly). If current_cpm = ₹200 → NO_CHANGE.
        → DECREASE_CPM only if cpm>200 and  roas_7d < 1.0 AND roas_15d < 1.5 AND roas_30d weak.
        → Otherwise: NO_CHANGE.
        -> If position = 1, then don't increase CPM. 
    
      Position 6–10:
        → PAUSE only if roas_7d < 1.0 AND roas_15d < 1.0 AND roas_30d < 1.0 (TIER 1 only).
        
        → DECREASE_CPM only if current_cpm > ₹200 (strictly). If current_cpm = ₹200 → NO_CHANGE.
    
        → DECREASE_CPM max 10% if CPM > ₹200.
    
      Position > 10:
        → Normal rules apply.
    
    ═══════════════════════════════════════════════════════════════
    STEP 5 — PREVIOUS RECOMMENDATION EVALUATION
    ═══════════════════════════════════════════════════════════════
    Read "previous_history" list from each keyword's own row for decision-making.
    Use "previous_summary" string only for writing the explanation — copy it verbatim.
    Do NOT cross-reference between keywords.
      SUCCESS (user_implemented = true, ROAS improved):
        → Confidence +0.10. Continue direction. Do not reverse without new evidence.
    
      FAILURE (user_implemented = true, ROAS declined):
        → Do NOT repeat same action.
        → INCREASE_CPM failed → NO_CHANGE or DECREASE_CPM.
        → DECREASE_CPM failed → NO_CHANGE.
        → Two consecutive INCREASE_CPM failures → NO_CHANGE for one full cycle.
    
      IGNORED (user_implemented = false):
        → Override produced better ROAS → align with user instinct, note it.
        → Override produced worse ROAS AND original was PAUSE → re-recommend PAUSE,
          confidence ≥ 0.92, flag urgency in explanation.
    
      UNKNOWN (user_implemented = null):
        → Treat as fresh. Apply Steps 1–4 normally.
        
    CRITICAL OVERRIDE RULE:
    If any later step changes the action, you MUST overwrite the action completely.
    No earlier action should remain in output, explanation, or JSON.
    Only FINAL_ACTION is allowed to appear anywhere.
    
    ═══════════════════════════════════════════════════════════════
    CONFIDENCE SCORING
    ═══════════════════════════════════════════════════════════════
      Base range : 0.70–0.85
      +0.10      : previous recommendation succeeded
      +0.05      : all sufficient windows agree on same direction
      -0.10      : only one window sufficient
      -0.05      : previous recommendation outcome unknown
      Maximum    : 0.95. Never output 1.0.
    
    
    PAUSE confidence scoring:
      If all three windows ROAS < 0.5: confidence 0.92
      If all three windows ROAS 0.5–0.99: confidence 0.90
      Add +0.05 if position > 50 (low position = low recovery potential)
      Add +0.05 if 30d spend > ₹3000 (significant budget already burned)
    
      
    
    ═══════════════════════════════════════════════════════════════
    ALTERNATIVE KEYWORD RULES (PAUSE action only)
    ═══════════════════════════════════════════════════════════════
      1. SEMANTIC RELEVANCE (mandatory): suggest only jewellery-related terms.
         ✗ Never suggest: flower, red, black, gift, color names, generic nouns
         ✓ If pausing "jhumka" → suggest "jhumki", "oxidised jhumka", "silver jhumka",
           "traditional earrings", "jhumki earrings"
         ✓ If pausing "earrings" → suggest "ear rings", "earring", "oxidised earrings",
           "silver earrings", "gold earrings"
    
      2. INTENT MATCH:
         Product type paused → suggest similar product types
         Style keyword paused → suggest same product with different style
         Brand paused → suggest category alternatives
    
      3. NO ACTIVE OVERLAP: never suggest a keyword already running in this campaign
    
      4. POOL CONSTRAINT: select only from provided KEYWORD POOL.
         If no relevant jewellery alternatives exist in pool → return []
    
      5. Maximum 5 per paused keyword.
    
    ═══════════════════════════════════════════════════════════════
    MANDATORY PRE-OUTPUT SCRATCHPAD
    ═══════════════════════════════════════════════════════════════
    Before writing the JSON array, write one decision line per keyword:
    
      [targeting] → TIER:[1/2/3] | GATE:[Strong/Moderate/Weak/N/A] |
      PAUSE_BLOCKED:[YES(reason) / NO] | FINAL_ACTION:[action]
      
      [targeting] → TIER:[1/2/3] | GATE:[Strong/Moderate/Weak/N/A] |
    PAUSE_BLOCKED:[YES(reason) / NO] | FINAL_ACTION:[action] | CONFIDENCE:[value]
    
    Examples:
      targeting → TIER:1 | GATE:Strong(15d 3.24≥3.0) | PAUSE_BLOCKED:YES(strong history) | FINAL_ACTION:NO_CHANGE
      targeting     → TIER:3 | GATE:N/A | PAUSE_BLOCKED:YES(TIER 3) | FINAL_ACTION:INCREASE_CPM
      targeting   → TIER:1 | GATE:Moderate(30d 2.38) | PAUSE_BLOCKED:YES(pos 5≤10) | FINAL_ACTION:DECREASE_CPM
    
    Complete ALL scratchpad lines before writing any JSON.
    JSON action MUST exactly match FINAL_ACTION in the scratchpad.
    If they differ → scratchpad wins. Fix the JSON before returning.
    
    ═══════════════════════════════════════════════════════════════
    OUTPUT FORMAT
    ═══════════════════════════════════════════════════════════════
    Output: scratchpad lines first, then the JSON array.
    
    Confidence should be mandatory
    cpm_change: percentage of CPM change.
    
    Each JSON object must contain ALL of these fields in this exact order:
    
    {{
      "campaign_id": "",
      "targeting": "",  
      "action": "INCREASE_CPM | DECREASE_CPM | PAUSE | NO_CHANGE",  
      "explanation": "",  
      "campaign_name": "",
      "cpm_change": 0,
      "confidence": 0.80,
      "alternative_keywords": [],
      "current_cpm":current_cpm,
      "campaign_budget":campaign_budget
    }}

    CONFIDENCE FIELD RULE — NON-NEGOTIABLE:
      "confidence" must be a decimal between 0.70 and 0.95.
      Valid examples: 0.70, 0.75, 0.80, 0.85, 0.90, 0.95.
      INVALID values: 0, 0.0, null, 1, 1.0, any value below 0.70.
      If you write 0 or 0.0, the output is rejected. Minimum is 0.70.
        CONFIDENCE OUTPUT RULE (mandatory, no exceptions):
      Every JSON object MUST contain a "confidence" field with a numeric value.
      Confidence is NEVER 0.0 unless explicitly calculated to be 0.0.
      If confidence calculation is skipped for any reason → default to 0.75.
      Omitting confidence or outputting null is a critical output error.
  
    
    
    
    Field rules:
      campaign_id         : use the campaign_id value from CURRENT DATA
      campaign_name       : use the campaign_name value from CURRENT DATA
      cpm_change          : 10 for INCREASE_CPM / DECREASE_CPM. 0 for NO_CHANGE / PAUSE.
      confidence : MANDATORY. Calculate using CONFIDENCE SCORING rules above.
             Output as decimal (e.g. 0.80, not 80). Never null. Never 0.0 as default.
             If unsure → floor is 0.70. Maximum is 0.95. Never 1.0.
             Omitting this field invalidates the entire JSON object.
    alternative_keywords: populated ONLY when action = PAUSE. Otherwise [].
      
      All 8 fields present per object: campaign_id, targeting, action, explanation, campaign_name,
        cpm_change, confidence, alternative_keywords,current_cpm,campaign_budget. No nulls. No missing keys. No extra keys.
    

    ═══════════════════════════════════════════════════════════════
    CPM FLOOR ENFORCEMENT (runs after every action decision — no exceptions)
    ═══════════════════════════════════════════════════════════════
    Before writing ANY JSON object, check this for EVERY keyword:

      IF action = DECREASE_CPM:
        → Read current_cpm exactly as given in the data.
        → Is current_cpm > 200? (strictly greater than, NOT equal to)
            YES (e.g. ₹201, ₹250, ₹300) → DECREASE_CPM is valid. Proceed.
            NO  (e.g. ₹200, ₹199, ₹150) → DECREASE_CPM is FORBIDDEN.
                Override action to NO_CHANGE immediately.
                Set cpm_change to 0.
                Do NOT write DECREASE_CPM anywhere in output.

      BOUNDARY CASES — memorize these exactly:
        current_cpm = ₹200 → NO_CHANGE (NOT DECREASE_CPM)
        current_cpm = ₹201 → DECREASE_CPM allowed
        current_cpm = ₹199 → NO_CHANGE (NOT DECREASE_CPM)

      This check overrides ALL previous steps.
      If DECREASE_CPM appears in your scratchpad but CPM ≤ ₹200 →
      fix the scratchpad FINAL_ACTION to NO_CHANGE before writing JSON.
      ═══════════════════════════════════════════════════════════════
        🚨 FINAL ACTION OVERRIDE — CPM HARD RULE (ABSOLUTE PRIORITY)
        ═══════════════════════════════════════════════════════════════

        This rule overrides ALL previous steps, gates, and decisions.

        For EVERY keyword, BEFORE writing the scratchpad:

        IF current_cpm ≤ 200:
          → FINAL_ACTION MUST BE NO_CHANGE
          → cpm_change MUST BE 0
          → DECREASE_CPM is strictly forbidden

        This is NOT a guideline. This is a hard override.

        Even if:
        - ROAS suggests decrease
        - Position is low
        - History is weak/moderate

        YOU MUST:
        → force FINAL_ACTION = NO_CHANGE

        ═══════════════════════════════════════════════════════════════
    
    ═══════════════════════════════════════════════════════════════
    FINAL JSON VALIDATION (run before returning — no exceptions)
    ═══════════════════════════════════════════════════════════════
    Before returning the JSON array, verify EVERY object contains ALL 8 fields:
      1. campaign_id       → string
      2. targeting         → string  
      3. action            → one of: INCREASE_CPM / DECREASE_CPM / NO_CHANGE / PAUSE. 
      4. explanation       → string (6-part pipe format)
      5. campaign_name     → string (from CURRENT DATA)
      6. cpm_change        → integer: 10 for INCREASE/DECREASE, 0 for NO_CHANGE/PAUSE
      7. confidence        → decimal between 0.70–0.95 (from scratchpad CONFIDENCE value)
      8. alternative_keywords → list ([] unless action = PAUSE)
      9.current_cpm 
      10.campaign_budget

    If ANY field is missing from ANY object → add it before returning.
    confidence must match the CONFIDENCE value written in the scratchpad line.
    Do not return until all 8 fields are present in every object.
    ═══════════════════════════════════════════════════════════════
    EXPLANATION FORMAT (mandatory — exact numbers only)
    ═══════════════════════════════════════════════════════════════
    Write the explanation as a single string in this exact 6-part format:
    
    7-Day: Spend ₹[exact]. ROAS [exact]. Position [exact].CPM ₹[current_cpm]. Sufficiency: [Met ₹500 / Not met ₹500]. | 15-Day: Spend ₹[exact]. ROAS [exact]. Sufficiency: [Met ₹1000 / Not met ₹1000]. Trend vs 7-day: [improving / declining / stable / insufficient]. | 30-Day: Spend ₹[exact]. ROAS [exact]. Sufficiency: [Met ₹2000 / Not met ₹2000]. Strength: [Strong ≥3.0 / Moderate 2.0–2.99 / Weak <2.0 / Insufficient data]. | Decision: [Exact condition. Which window drove it. Exact numbers. Max 2 sentences.] | 
    Previous: [copy previous_summary from this keyword's row verbatim — do not paraphrase or shorten] | AI Recommendation: [action] — [1 sentence rationale with exact numbers]. User Options: (1) NO_CHANGE — [trade-off]; (2) INCREASE_CPM — [trade-off]; (3) DECREASE_CPM — [trade-off]; (4) PAUSE — [trade-off or "Not applicable: reason"]. Include only feasible options. Always include at least 2.
    
    STRENGTH LABEL RULE:
      Only label strength for spend-sufficient windows.
      Insufficient window → always write "Insufficient data" regardless of ROAS.
      Example: 30d ROAS = 3.63 but spend ₹1937 < ₹2000 → "Insufficient data" not "Strong".
    
    BANNED WORDS: approximately, around, roughly, borderline, generally, varies,
    seems, appears, likely performing, near.
    No internal step names in explanations (Step 1, Rule A, TIER 3, etc.).
    State conclusions with exact numbers only.
    
    ═══════════════════════════════════════════════════════════════
    ANALYSIS WORKFLOW (execute in this exact order, every keyword)
    ═══════════════════════════════════════════════════════════════
     1. Read keyword_spend_7d, 15d, 30d. Classify TIER 1 / 2 / 3.
     2. TIER 3 → INCREASE_CPM. Write scratchpad. Skip to step 11.
     3. TIER 2 → note sufficient windows. PAUSE blocked.
     4. Check roas_30d then roas_15d (sufficient windows, hard line 3.0).
     5. HISTORY GATE fires (≥ 3.0):
          Check SCALE conditions. Met → INCREASE_CPM. Else → NO_CHANGE.
          Write scratchpad. Skip to step 11.
     6. Moderate history (2.0–2.99): DECREASE_CPM unless prev failed → NO_CHANGE.
     7. Weak history: Rule A → B → C → D. First match only.
     8. Position gate: position ≤ 10 and action = PAUSE → override now.
     9. Strong history self-check: action = DECREASE_CPM or PAUSE and strong
        history confirmed → override to NO_CHANGE or INCREASE_CPM before output.
    10. CPM floor check: action = DECREASE_CPM and CPM ≤ ₹200 → NO_CHANGE.
    11. Previous rec evaluation. Apply learning from user overrides.
    12. Confidence calculation.
    13. Write scratchpad line (if not already written).
    14. Write explanation in mandatory 6-part pipe-separated format.
    15. After ALL scratchpad lines complete → write JSON array.
    16. Final check: each JSON action = scratchpad FINAL_ACTION.
        Any mismatch → fix JSON before returning.
        
    """
    
    
    # ============================================================
    # MAIN PIPELINE
    # ============================================================
    
    # Normalize campaign_name_map keys to string — prevents int/str mismatch
    campaign_name_map_str = {str(k): v for k, v in campaign_name_map.items()}
    
    all_suggestions = []
    
    for campaign_id, campaign_group_df in aggregated_df.groupby("campaign_id"):
        
        suggested_keyword_query = f"""
        SELECT 
            ks.suggested_value,
            MAX(ks.keyword_searches) AS total_searches,
            MAX(ks.weighted_score) AS weighted_score,
            BOOL_OR(ks.is_brand_keyword) AS is_brand
        FROM voylla."Blinkit_keyword_suggestions" ks
        JOIN "voylla"."Blinkit_CampaignWise_ProductID" bcp
            ON ks.product_id = bcp.product_id
        WHERE bcp.campaign_id = '{campaign_id}'
        GROUP BY ks.suggested_value
        ORDER BY weighted_score DESC;
        """
    
        keyword_pool_df = pd.read_sql(suggested_keyword_query, engine)
    #     print(keyword_pool_df)
    
        if campaign_id not in sufficient_campaigns:
            continue
    
        campaign_id_str = str(campaign_id)
        campaign_name   = campaign_name_map_str.get(campaign_id_str, "")
    
        print(f"\n🚀 Processing campaign {campaign_id_str} — {campaign_name}")
    
        data_for_llm = campaign_group_df.to_dict(orient="records")
    
        # inject campaign_name into each row so LLM can read it directly
        for row in data_for_llm:
            row["campaign_id"]   = campaign_id_str
            row["campaign_name"] = campaign_name
            targeting_key = str(row.get("targeting", "")).strip().lower().replace(" ", "_")
            prev = build_previous_context(history_df, campaign_id_str, targeting_key)
    
            row["previous_summary"] = prev["previous_summary"]
    
            row["previous_history"] = prev["previous_history"]

#         campaign_history_df = history_df[
#             history_df["campaign_id"].astype(str) == campaign_id_str
#         ].copy()
        
#         print(campaign_history_df)
    
        user_message = f"""
        ⚠️ HARD RULE (non-negotiable): If current_cpm ≤ 200, action = NO_CHANGE. DECREASE_CPM is forbidden at CPM ≤ 200. This applies to every keyword below without exception.

        -> Each row contains two fields:
           "previous_summary" → copy this verbatim into the explanation "Previous:" section.
           "previous_history" → use this list for Step 5 decision-making (newest first).
        -> Do NOT repeat an action flagged with ⚠️ LOOP in previous_summary.
        -> Do NOT cross-reference history between keywords.
        
        CURRENT DATA (7-day aggregated with 15-day and 30-day ROAS and spend):
        {json.dumps(data_for_llm, indent=2)}
        

        KEYWORD POOL (for alternative keywords when action = PAUSE):
        {json.dumps(keyword_pool_df.to_dict(orient='records'), indent=2)}

        Write scratchpad lines first. Then return ONLY the JSON array.
        No preamble, no commentary, no markdown fences.

        Instructions:
        If current_cpm is <200 and action is decrease_cpm , then action should be no change 
             - For each keyword, read "previous_context" from the row itself.
             - Use it in the explanation under the "Previous:" section.
             
        1. Write scratchpad lines first (one per keyword).
        2. Then output the JSON array with ALL 8 fields per object:
           campaign_id, targeting, action, explanation, campaign_name,
           cpm_change, confidence, alternative_keywords,current_cpm,campaign_budget.
        3. confidence must match the CONFIDENCE value from your scratchpad line.
        4. No missing fields. No null values. No markdown fences.
        
        """

        raw_response = ""
    
        try:
            response = llm.invoke([
                SystemMessage(content=SYSTEM_PROMPT),
                HumanMessage(content=user_message)
            ] , config={"max_tokens": 8000})
    
            raw_response = response.content
            print(f"📝 Raw response preview:\n{raw_response[:600]}\n...")
    
            suggestion_response = extract_json(raw_response)
            print("🔍 Sample JSON row:", json.dumps(suggestion_response[0], indent=2))
    
            # patch campaign_id and campaign_name on every row — guaranteed correct
            # preserve all other columns exactly as returned by LLM
            for row in suggestion_response:
                row["campaign_id"]   = campaign_id_str
                row["campaign_name"] = campaign_name
    
            print(f"✅ {len(suggestion_response)} keywords processed")
    
            all_suggestions.extend(suggestion_response)
    
            save_llm_action(engine, suggestion_response,brand)
    
        except ValueError as e:
            print(f"❌ JSON parse error for campaign {campaign_id_str}: {e}")
            if raw_response:
                print(f"   Raw response snippet: {raw_response[:400]}")
            continue
    
        except Exception as e:
            print(f"❌ Unexpected error for campaign {campaign_id_str}: {e}")
            if raw_response:
                print(f"   Raw response snippet: {raw_response[:400]}")
            continue
    
    print(f"\n📊 Total keywords processed: {len(all_suggestions)}")
    
#     ─────────────────────────────────────────────
#     STEP 1: Handle INSUFFICIENT campaigns
#     (just for alternative keyword suggestions)
#     ─────────────────────────────────────────────
    for campaign_id in insufficient_campaigns:
        
        campaign_group_df = aggregated_df[aggregated_df["campaign_id"] == campaign_id]
        campaign_name = campaign_name_map_str.get(str(campaign_id), "")
        data_for_llm = campaign_group_df.to_dict(orient="records")
    #     print(data_for_llm)
        total_spend = campaign_spend_df[
            campaign_spend_df["Campaign ID"] == campaign_id
        ]["campaign_spend"].values[0]
        
        
        
        suggested_keyword_query = f"""
        SELECT 
            ks.suggested_value,
            MAX(ks.keyword_searches) AS total_searches,
            MAX(ks.weighted_score) AS weighted_score,
            BOOL_OR(ks.is_brand_keyword) AS is_brand
        FROM voylla."Blinkit_keyword_suggestions" ks
        JOIN "voylla"."Blinkit_CampaignWise_ProductID" bcp
            ON ks.product_id = bcp.product_id
        WHERE bcp.campaign_id = '{campaign_id}'
        GROUP BY ks.suggested_value
        ORDER BY weighted_score DESC;
        """
    
        keyword_pool_df = pd.read_sql(suggested_keyword_query, engine)
    #     print(keyword_pool_df)
    
        print(f"\n⚠️  INSUFFICIENT campaign {campaign_id} | 7d campaign spend ₹{total_spend:.0f} — generating keyword suggestions only")
    
        insufficient_prompt = f"""
        You are a performance marketing expert analyzing Instamart ad campaigns.
    
        CONTEXT:
        - This campaign has a total 7-day spend of ₹{total_spend:.0f}, which is below the ₹500 threshold.
        - All keywords in this campaign are classified as INSUFFICIENT DATA.
        - Do NOT recommend PAUSE or INCREASE_CPM. Action must always be INSUFFICIENT_DATA.
        - Your only task: suggest 2–3 semantically similar alternative keywords for EACH keyword below.
    
        RULES FOR ALTERNATIVE KEYWORDS:
        1. Suggest keywords semantically similar to the targeting keyword
        2. DO NOT suggest the same keyword being analyzed
        3. Each keyword MUST receive DIFFERENT alternative suggestions
        4. No duplicates across the entire batch
        5. If similarity is low, still suggest the closest 2 keywords from the keyword pool. Never return an empty array.
    
        OUTPUT FORMAT:
        Return ONLY a valid JSON array. Each object must have this exact structure:
        
          "campaign_id": "{campaign_id}",
          "campaign_name": "{campaign_name}",
          "targeting": "keyword name",
          "action": "INSUFFICIENT_DATA",
          "cpm_change": null,
          "confidence": 0.5,
          "explanation": "7-day campaign spend ₹{total_spend:.0f}. Below ₹500 threshold. INSUFFICIENT data. Action deferred — suggesting alternative keywords to explore.",
          "alternative_keywords": ["kw1", "kw2", "kw3","k4"],
          "current_cpm":current_cpm,
          "campaign_budget":campaign_budget
          
        KEYWORD POOL (for alternatives keywords):
        {keyword_pool_df.to_dict(orient='records')}
        
    
        CURRENT KEYWORDS IN THIS CAMPAIGN:
        {json.dumps(data_for_llm, indent=2)}
    
        Return ONLY the JSON array.

        Instructions:
        1. Write scratchpad lines first (one per keyword).
        2. Then output the JSON array with ALL fields per object:
           campaign_id, targeting, action, explanation, campaign_name,
           cpm_change, confidence, alternative_keywords,current_cpm,campaign_budget.
        3. confidence must match the CONFIDENCE value from your scratchpad line.
        4. No missing fields. No null values. No markdown fences.
        """
    
        response = llm.invoke(insufficient_prompt).content
    #     print(response)
        suggestion_response = extract_json(response)
    
        for row in suggestion_response:
            if not isinstance(row, dict):
                continue 
            row["campaign_id"] = campaign_id
            row["campaign_name"] = campaign_name
            row["action"] = "INSUFFICIENT DATA"  # force correct action
            row["explanation"]="INSUFFICIENT DATA"
    
    #     print(suggestion_response)
        save_llm_action(engine, suggestion_response,brand)
        all_suggestions.extend(suggestion_response)

    
    
    if all_suggestions:
        import pandas as pd
        results_df = pd.DataFrame(all_suggestions)
    
        print("\n📋 Action distribution:")
        print(results_df["action"].value_counts().to_string())
    
        print("\n📋 Campaign name check (should have no blanks):")
        blank_names = results_df[results_df["campaign_name"] == ""]
        
        if blank_names.empty:
            print("   ✅ All campaign names populated")
        else:
            print(f"   ⚠️  {len(blank_names)} rows with blank campaign name:")
            print(blank_names[["campaign_id", "targeting"]].to_string())
            
    

[421755, 296466, 401219, 296465, 296464, 302014, 436744, 288219, 418695, 398499, 364463, 364418, 398462, 296470, 428706, 302008, 436746, 422337, 301223, 163795, 421774, 296468, 359616, 301236, 301245, 436753]
✅ Sufficient campaigns (≥₹500): 24
⚠️  Insufficient campaigns (<₹500): 2

🚀 Processing campaign 163795 — Men's Gifting (Cuffling)


In [ ]:
import datetime
st=datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S')

#Script Details
SD={
    'Script_Name':'Blinkit_actions_llm_marketing.ipynb',
    'Output':'Table',
    'Table Name':'"Blinkit_actions_llm"',
    'Sheet_id':'-',
    'Report_Name':'-',
    'Updated_at':st
}

SD = pd.DataFrame([SD])

SD.to_sql('python_log', engine, schema='voylla', if_exists='append', index=False)